# 🌟 Datathon Passos Mágicos — Análise Completa de Dados

> **Projeto:** Datathon FIAP PosTech × ONG Passos Mágicos  
> **Objetivo:** Análise exploratória, construção de features e modelagem preditiva para identificar alunos em risco de defasagem escolar.  
> **Equipe:** FIAP PosTech  
> **Data:** Abril / 2026

---

## 📌 Sobre o Projeto

A ONG **Passos Mágicos** atende centenas de crianças e jovens em situação de vulnerabilidade social na Grande São Paulo, oferecendo educação complementar e suporte psicossocial. O objetivo deste Datathon é usar os dados históricos da ONG para:

1. **Entender o perfil de defasagem escolar** dos alunos ao longo dos anos (2020–2024)  
2. **Identificar padrões** nos indicadores pedagógicos e psicossociais  
3. **Construir um modelo preditivo** que aponte alunos em risco de defasagem, auxiliando a equipe da ONG a intervir proativamente  

---

## 📁 Estrutura do Projeto

```
Datathon/
├── data/
│   ├── PEDE_PASSOS_DATASET_FIAP.csv          ← Dados longitudinais 2020-2022
│   ├── BASE DE DADOS PEDE 2024 - DATATHON.xlsx  ← Dados 2024 (base principal)
│   └── Base de dados - Passos Mágicos/       ← Dados históricos adicionais
├── src/
│   ├── data_loader.py   ← Carregamento, limpeza e preparação dos dados
│   ├── analysis.py      ← Funções de análise e visualização (11 perguntas)
│   └── model.py         ← Classe ModeloRiscoDefasagem (ML pipeline)
├── models/
│   └── modelo_risco.joblib  ← Modelo treinado serializado
├── train_model.py       ← Script de treinamento standalone
├── app.py               ← Aplicação Streamlit interativa
└── notebooks/           ← Este notebook
```

---

## 🗂️ Índice

1. [Configuração do Ambiente](#1-configuração-do-ambiente)
2. [Carregamento e Limpeza dos Dados](#2-carregamento-e-limpeza-dos-dados)
3. [Criação do Dataset Longitudinal](#3-criação-do-dataset-longitudinal)
4. [Análise 1: Adequação ao Nível (IAN)](#4-análise-1-adequação-ao-nível-ian)
5. [Análise 2: Desempenho Acadêmico (IDA)](#5-análise-2-desempenho-acadêmico-ida)
6. [Análise 3: Engajamento vs Desempenho](#6-análise-3-engajamento-vs-desempenho)
7. [Análise 4: Autoavaliação (IAA) vs Desempenho Real](#7-análise-4-autoavaliação-iaa-vs-desempenho-real)
8. [Análise 5: Aspectos Psicossociais (IPS)](#8-análise-5-aspectos-psicossociais-ips)
9. [Análise 6: Aspectos Psicopedagógicos (IPP)](#9-análise-6-aspectos-psicopedagógicos-ipp)
10. [Análise 7: Ponto de Virada (IPV)](#10-análise-7-ponto-de-virada-ipv)
11. [Análise 8: Multidimensionalidade dos Indicadores](#11-análise-8-multidimensionalidade-dos-indicadores)
12. [Análise 9: Efetividade do Programa](#12-análise-9-efetividade-do-programa)
13. [Análise 10: Insights Adicionais](#13-análise-10-insights-adicionais)
14. [Preparação dos Dados para Modelagem](#14-preparação-dos-dados-para-modelagem)
15. [Treinamento e Avaliação dos Modelos](#15-treinamento-e-avaliação-dos-modelos)
16. [Predição Individual de Risco](#16-predição-individual-de-risco)
17. [Conclusões e Próximos Passos](#17-conclusões-e-próximos-passos)

---
## 1. Configuração do Ambiente

### 1.1 Glossário dos Indicadores

| Indicador | Nome Completo | Descrição |
|-----------|---------------|------------------------------|
| **INDE** | Índice de Desenvolvimento Educacional | Índice composto que resume o desempenho geral do aluno |
| **IAA** | Índice de Autoavaliação | Autoavaliação do aluno sobre seu próprio desempenho |
| **IEG** | Índice de Engajamento | Mede o nível de envolvimento e participação do aluno |
| **IPS** | Índice Psicossocial | Avalia aspectos emocionais e sociais do aluno |
| **IDA** | Índice de Desempenho Acadêmico | Desempenho em notas e avaliações escolares |
| **IPP** | Índice Psicopedagógico | Avaliação psicopedagógica do aluno |
| **IPV** | Índice de Ponto de Virada | Indicador da evolução transformadora do aluno |
| **IAN** | Índice de Adequação ao Nível | Mede se o aluno está no nível esperado para sua fase |
| **FASE** | Fase / Nível | Nível atual do aluno no programa (escala própria da PM) |
| **PEDRA** | Classificação por Pedra | Quartzo / Ágata / Ametista / Topázio (baseado no INDE) |

### 1.2 Regra de Classificação por Pedra (baseada no INDE)

```
INDE < 5.506   → Quartzo   (nível mais inicial)
5.506 ≤ INDE < 6.868 → Ágata
6.868 ≤ INDE < 8.230 → Ametista
INDE ≥ 8.230   → Topázio   (nível de excelência)
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 1 — Imports e configurações globais
#
# Importamos todas as bibliotecas necessárias para análise de dados,
# visualização e modelagem preditiva.
#
# Bibliotecas principais:
#   • pandas / numpy  → manipulação e análise de dados
#   • plotly          → visualizações interativas
#   • scikit-learn    → pré-processamento e modelos de ML
#   • xgboost         → modelo de gradient boosting otimizado
#   • joblib          → serialização de modelos
# ─────────────────────────────────────────────────────────────

import sys
import os
import warnings

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings('ignore')

# Garantir que o diretório raiz do projeto está no PATH
# para que os módulos src.* possam ser importados corretamente
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"✅ Ambiente configurado com sucesso!")
print(f"   Python: {sys.version.split()[0]}")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy:  {np.__version__}")
print(f"   Diretório raiz: {ROOT}")

---
## 2. Carregamento e Limpeza dos Dados

O projeto trabalha com **duas fontes de dados** distintas, correspondentes a períodos diferentes:

### 📦 Fonte 1 — CSV Longitudinal (2020–2022)
Arquivo: `data/PEDE_PASSOS_DATASET_FIAP.csv`

- Formato: **wide** — cada aluno tem uma linha, com colunas como `INDE_2020`, `INDE_2021`, `INDE_2022`
- Separador: ponto e vírgula (`;`), encoding `latin1`
- Problemas identificados e corrigidos:
  - Indicadores numéricos armazenados como `str` → convertidos com `pd.to_numeric(errors='coerce')`
  - Encoding corrompido nas colunas PEDRA: `'Ã\x81gata'` → `'Ágata'`, `'TopÃ¡zio'` → `'Topázio'`
  - Valores inválidos `'#NULO!'` e strings como `'D9600'` → substituídos por `NaN`
  - FASE extraída de `FASE_TURMA_2020` (primeiro caractere numérico)

### 📦 Fonte 2 — XLSX 2024 (Base principal)
Arquivo: `data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx`

- Base mais completa, com informações de avaliadores, bolsas, defasagem calculada
- Colunas renomeadas para padronização (ex: `'Fase'` → `'FASE'`, `'Matem'` → `'NOTA_MAT'`)
- Feature derivada: `ANOS_PM = 2022 - ANO_INGRESSO` (tempo de permanência na ONG)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 2 — Funções de carregamento de dados
#
# Reproduzimos aqui, de forma documentada, as funções definidas
# em src/data_loader.py para total transparência e executabilidade
# sem dependência de imports relativos.
# ─────────────────────────────────────────────────────────────

def carregar_dataset_csv(caminho='../data/PEDE_PASSOS_DATASET_FIAP.csv'):
    """
    Carrega o dataset CSV com dados longitudinais 2020-2022.

    Etapas:
      1. Leitura com separador ';' e encoding 'latin1' (padrão de exportações BR)
      2. Conversão dos indicadores numéricos de string → float (coerce = NaN se inválido)
      3. Correção de encoding UTF-8 corrompido nas colunas de pedra
      4. Limpeza de valores inválidos nas colunas de Ponto de Virada
      5. Extração da FASE do campo FASE_TURMA_2020
    """
    df = pd.read_csv(caminho, sep=';', encoding='latin1')

    # Lista de todos os indicadores por ano
    indicadores_2020 = ['INDE_2020', 'IAA_2020', 'IEG_2020', 'IPS_2020',
                        'IDA_2020', 'IPP_2020', 'IPV_2020', 'IAN_2020']
    indicadores_2021 = ['INDE_2021', 'IAA_2021', 'IEG_2021', 'IPS_2021',
                        'IDA_2021', 'IPP_2021', 'IPV_2021', 'IAN_2021']
    indicadores_2022 = ['INDE_2022', 'IAA_2022', 'IEG_2022', 'IPS_2022',
                        'IDA_2022', 'IPP_2022', 'IPV_2022', 'IAN_2022',
                        'NOTA_PORT_2022', 'NOTA_MAT_2022', 'NOTA_ING_2022']

    # Conversão numérica — strings inválidas viram NaN (errors='coerce')
    for col in indicadores_2020 + indicadores_2021 + indicadores_2022:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Correção de encoding nas colunas de PEDRA
    for col in ['PEDRA_2020', 'PEDRA_2021', 'PEDRA_2022']:
        if col in df.columns:
            df[col] = df[col].replace({
                'Ã\x81gata': 'Ágata',
                'TopÃ¡zio': 'Topázio',
                '#NULO!': np.nan,
                'D9891/2A': np.nan
            })

    # Limpeza de Ponto de Virada
    for col in ['PONTO_VIRADA_2020', 'PONTO_VIRADA_2021', 'PONTO_VIRADA_2022']:
        if col in df.columns:
            df[col] = df[col].replace({
                'NÃ£o': 'Não',
                '#NULO!': np.nan,
                'D9600': np.nan
            })

    # Extração da FASE numérica de 'FASE_TURMA_2020' (ex: '3A' → 3)
    if 'FASE_TURMA_2020' in df.columns:
        df['FASE_2020'] = df['FASE_TURMA_2020'].apply(
            lambda x: int(x[0]) if pd.notna(x) and len(str(x)) > 0 and str(x)[0].isdigit() else np.nan
        )

    return df


def carregar_dataset_xlsx(caminho='../data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx'):
    """
    Carrega e padroniza o dataset XLSX de 2024.

    Etapas:
      1. Leitura do Excel
      2. Renomeação de colunas para o padrão SNAKE_UPPER do projeto
      3. Criação da feature ANOS_PM (tempo na ONG)
    """
    df = pd.read_excel(caminho)

    rename_map = {
        'Fase': 'FASE',          'Turma': 'TURMA',
        'Nome': 'NOME',           'Ano nasc': 'ANO_NASC',
        'Idade 22': 'IDADE',      'Gênero': 'GENERO',
        'Ano ingresso': 'ANO_INGRESSO',
        'Instituição de ensino': 'INSTITUICAO_ENSINO',
        'Pedra 20': 'PEDRA_2020', 'Pedra 21': 'PEDRA_2021',
        'Pedra 22': 'PEDRA_2022', 'INDE 22': 'INDE',
        'Cg': 'CG',               'Cf': 'CF',             'Ct': 'CT',
        'Nº Av': 'NUM_AVALIADORES',
        'Avaliador1': 'AVALIADOR_1', 'Rec Av1': 'REC_AVAL_1',
        'Avaliador2': 'AVALIADOR_2', 'Rec Av2': 'REC_AVAL_2',
        'Avaliador3': 'AVALIADOR_3', 'Rec Av3': 'REC_AVAL_3',
        'Avaliador4': 'AVALIADOR_4', 'Rec Av4': 'REC_AVAL_4',
        'Rec Psicologia': 'REC_PSICOLOGIA',
        'Matem': 'NOTA_MAT',      'Portug': 'NOTA_PORT',
        'Inglês': 'NOTA_ING',     'Indicado': 'INDICADO_BOLSA',
        'Atingiu PV': 'PONTO_VIRADA', 'Fase ideal': 'FASE_IDEAL',
        'Defas': 'DEFASAGEM',
        'Destaque IEG': 'DESTAQUE_IEG',
        'Destaque IDA': 'DESTAQUE_IDA',
        'Destaque IPV': 'DESTAQUE_IPV',
    }
    df = df.rename(columns=rename_map)

    # Feature de engajamento temporal: quantos anos o aluno está na PM
    df['ANOS_PM'] = 2022 - df['ANO_INGRESSO']

    return df


print("✅ Funções de carregamento definidas.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 3 — Carregamento efetivo dos dados
# ─────────────────────────────────────────────────────────────

print("📊 Carregando dados...\n")

# Dataset longitudinal 2020-2022
df_csv = carregar_dataset_csv()
print(f"✅ CSV carregado: {df_csv.shape[0]} alunos × {df_csv.shape[1]} colunas")

# Dataset 2024 (base principal)
df_xlsx = carregar_dataset_xlsx()
print(f"✅ XLSX carregado: {df_xlsx.shape[0]} alunos × {df_xlsx.shape[1]} colunas")

print("\n📋 Primeiras linhas do CSV (formato wide):")
df_csv.head(3)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 4 — Exploração inicial do dataset 2024
# ─────────────────────────────────────────────────────────────

print("📋 Colunas do dataset 2024:")
print(list(df_xlsx.columns))

print("\n📊 Estatísticas descritivas dos principais indicadores:")
indicadores_principais = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']
cols_available = [c for c in indicadores_principais if c in df_xlsx.columns]
df_xlsx[cols_available].describe().round(3)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 5 — Análise de valores faltantes
#
# Identificar o percentual de NaN em cada coluna é fundamental
# para decisões de imputação e feature selection.
# ─────────────────────────────────────────────────────────────

print("🔍 Porcentagem de valores faltantes por coluna (dataset CSV):")
nulos_csv = (df_csv.isnull().sum() / len(df_csv) * 100).sort_values(ascending=False)
nulos_csv = nulos_csv[nulos_csv > 0]
print(nulos_csv.round(1).to_string())

print("\n🔍 Porcentagem de valores faltantes (dataset XLSX 2024):")
nulos_xlsx = (df_xlsx.isnull().sum() / len(df_xlsx) * 100).sort_values(ascending=False)
nulos_xlsx = nulos_xlsx[nulos_xlsx > 0]
print(nulos_xlsx.round(1).to_string())

---
## 3. Criação do Dataset Longitudinal

Para analisar a **evolução temporal** dos alunos, precisamos converter o formato **wide** (um aluno por linha, colunas por ano) para o formato **long** (uma observação por aluno-ano).

### Estratégia de união:
- **2020, 2021, 2022** → extraídos do CSV (formato wide → explodidos por ano)
- **2024** → extraído do XLSX e adicionado como ano adicional

O resultado é um dataframe unificado com a coluna `ANO`, permitindo análises de time series por aluno e por grupo.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 6 — Construção do dataset longitudinal (formato long)
#
# Função `criar_dataset_longitudinal` do src/data_loader.py
#
# Para cada aluno e cada ano, verificamos se o INDE está disponível.
# Caso positivo, criamos um registro com todos os indicadores daquele ano.
# Isso permite comparar a mesma métrica (ex: IDA) em anos distintos.
# ─────────────────────────────────────────────────────────────

def criar_dataset_longitudinal(df_csv, df_xlsx=None):
    """
    Transforma o CSV wide em formato long (um registro por aluno-ano).
    Opcionalmente integra dados do XLSX de 2024 como mais um ano.

    Retorna um DataFrame com colunas:
      NOME, ANO, INDE, IAA, IEG, IPS, IDA, IPP, IPV, IAN,
      PEDRA, PONTO_VIRADA, FASE
    """
    registros = []
    indicadores = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']

    # ── Anos 2020-2022 (CSV) ──
    for ano in [2020, 2021, 2022]:
        pedra_col = f'PEDRA_{ano}'
        pv_col    = f'PONTO_VIRADA_{ano}'
        fase_col  = f'FASE_{ano}' if f'FASE_{ano}' in df_csv.columns else None

        for idx, row in df_csv.iterrows():
            inde_val = row.get(f'INDE_{ano}')
            if pd.isna(inde_val):
                continue   # Aluno sem dados neste ano → pular

            registro = {'NOME': str(row.get('NOME', '')).upper(), 'ANO': ano}

            for ind in indicadores:
                col_name = f'{ind}_{ano}'
                if col_name in df_csv.columns:
                    registro[ind] = row[col_name]

            if pedra_col in df_csv.columns: registro['PEDRA']        = row[pedra_col]
            if pv_col    in df_csv.columns: registro['PONTO_VIRADA'] = row[pv_col]
            if fase_col  and fase_col in df_csv.columns:
                                             registro['FASE']         = row[fase_col]

            registros.append(registro)

    # ── Ano 2024 (XLSX) ──
    if df_xlsx is not None:
        for idx, row in df_xlsx.iterrows():
            inde_val = row.get('INDE')
            if pd.isna(inde_val):
                continue

            registro = {'NOME': str(row.get('NOME', '')).upper(), 'ANO': 2024}

            for ind in indicadores:
                if ind in df_xlsx.columns:
                    registro[ind] = row[ind]

            if 'PEDRA_2022' in df_xlsx.columns: registro['PEDRA']        = row['PEDRA_2022']
            if 'PONTO_VIRADA' in df_xlsx.columns: registro['PONTO_VIRADA'] = row['PONTO_VIRADA']
            if 'FASE'         in df_xlsx.columns: registro['FASE']         = row['FASE']

            registros.append(registro)

    return pd.DataFrame(registros)


# Criação do dataset longitudinal
df_long = criar_dataset_longitudinal(df_csv, df_xlsx)

print(f"✅ Dataset longitudinal criado!")
print(f"   Total de registros (aluno-ano): {len(df_long)}")
print(f"   Anos disponíveis: {sorted(df_long['ANO'].unique())}")
print(f"\n📋 Primeiras linhas:")
df_long.head()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 7 — Funções auxiliares de classificação
#
# classificar_risco_ian → categoriza alunos por risco baseado no IAN
# classificar_pedra    → categoriza por "pedra" baseado no INDE
# ─────────────────────────────────────────────────────────────

def classificar_risco_ian(valor_ian):
    """
    Classifica o risco de defasagem baseado no valor IAN.

    Lógica dos limiares:
      IAN >= 8     → Adequado (aluno no ou acima do nível esperado)
      5 <= IAN < 8 → Moderadamente defasado
      2.5 <= IAN < 5 → Severamente defasado
      IAN < 2.5    → Criticamente defasado
    """
    if pd.isna(valor_ian):  return 'Sem dados'
    if valor_ian >= 8:      return 'Adequado'
    if valor_ian >= 5:      return 'Moderadamente defasado'
    if valor_ian >= 2.5:    return 'Severamente defasado'
    return 'Criticamente defasado'


def classificar_pedra(inde):
    """
    Classifica o aluno por pedra preciosa baseado no INDE.

    Limiares definidos pela Passos Mágicos:
      INDE < 5.506          → Quartzo
      5.506 <= INDE < 6.868 → Ágata
      6.868 <= INDE < 8.230 → Ametista
      INDE >= 8.230          → Topázio
    """
    if pd.isna(inde): return 'Sem classificação'
    if inde < 5.506:  return 'Quartzo'
    if inde < 6.868:  return 'Ágata'
    if inde < 8.230:  return 'Ametista'
    return 'Topázio'


# Aplicar classificações ao dataset longitudinal
df_long['RISCO_IAN'] = df_long['IAN'].apply(classificar_risco_ian)
df_long['PEDRA_CALC'] = df_long['INDE'].apply(classificar_pedra)

print("✅ Classificações aplicadas!")
print("\n📊 Distribuição do RISCO_IAN (todos os anos):")
print(df_long['RISCO_IAN'].value_counts())

---
## 4. Análise 1: Adequação ao Nível (IAN)

O **IAN (Índice de Adequação ao Nível)** é o principal indicador de defasagem escolar no programa. Ele mede se o aluno está no nível esperado para sua fase/série.

**Perguntas respondidas:**
- Qual o perfil geral de defasagem da base?
- O percentual de alunos em risco está aumentando ou diminuindo ao longo dos anos?
- Qual a evolução da média e mediana do IAN?

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 8 — Análise da evolução do IAN por ano
#
# Equivalente à função `analise_ian` em src/analysis.py
# ─────────────────────────────────────────────────────────────

CORES_RISCO = {
    'Adequado': '#2ECC71',
    'Moderadamente defasado': '#F39C12',
    'Severamente defasado': '#E74C3C',
    'Criticamente defasado': '#C0392B',
    'Sem dados': '#95A5A6'
}

CORES_PEDRAS = {
    'Quartzo': '#9B59B6',
    'Ágata': '#3498DB',
    'Ametista': '#8E44AD',
    'Topázio': '#F39C12'
}

TEMPLATE = 'plotly_white'

# ── Distribuição de risco por ano ──
dist = df_long.groupby(['ANO', 'RISCO_IAN']).size().reset_index(name='CONTAGEM')
totais = df_long.groupby('ANO').size().reset_index(name='TOTAL')
dist = dist.merge(totais, on='ANO')
dist['PERCENTUAL'] = (dist['CONTAGEM'] / dist['TOTAL'] * 100).round(1)

fig_ian = px.bar(
    dist, x='ANO', y='PERCENTUAL', color='RISCO_IAN',
    color_discrete_map=CORES_RISCO,
    title='📊 Evolução da Defasagem dos Alunos (IAN) por Ano',
    labels={'PERCENTUAL': 'Percentual (%)', 'ANO': 'Ano', 'RISCO_IAN': 'Classificação'},
    barmode='group',
    template=TEMPLATE,
    text='PERCENTUAL'
)
fig_ian.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_ian.show()

print("\n📋 Tabela: Percentual por classificação de risco e ano:")
dist.pivot_table(values='PERCENTUAL', index='RISCO_IAN', columns='ANO').round(1)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 9 — Evolução da média e mediana do IAN
# ─────────────────────────────────────────────────────────────

media_ian = df_long.groupby('ANO')['IAN'].agg(['mean', 'median', 'std']).reset_index()

fig_ian_media = go.Figure()
fig_ian_media.add_trace(go.Scatter(
    x=media_ian['ANO'], y=media_ian['mean'],
    mode='lines+markers+text',
    text=media_ian['mean'].round(2),
    textposition='top center',
    name='Média IAN',
    line=dict(color='#E74C3C', width=3),
    marker=dict(size=12)
))
fig_ian_media.add_trace(go.Scatter(
    x=media_ian['ANO'], y=media_ian['median'],
    mode='lines+markers',
    name='Mediana IAN',
    line=dict(color='#3498DB', width=3, dash='dash'),
    marker=dict(size=12)
))
fig_ian_media.update_layout(
    title='📈 Evolução da Média e Mediana do IAN ao Longo dos Anos',
    xaxis_title='Ano', yaxis_title='IAN',
    template=TEMPLATE
)
fig_ian_media.show()

print("\n📊 Estatísticas do IAN por ano:")
media_ian.round(3)

---
## 5. Análise 2: Desempenho Acadêmico (IDA)

O **IDA (Índice de Desempenho Acadêmico)** mede diretamente o rendimento escolar do aluno em provas e avaliações. 

**Perguntas respondidas:**
- O desempenho médio está melhorando ao longo dos anos?
- Como o IDA evolui por classificação de pedra?
- Qual o desvio padrão? (dispersão da turma)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 10 — Análise de Desempenho Acadêmico (IDA)
#
# Equivalente à função `analise_ida` em src/analysis.py
# ─────────────────────────────────────────────────────────────

media_ida = df_long.groupby('ANO')['IDA'].agg(['mean', 'median', 'std']).reset_index()

# Gráfico dual-axis: média IDA no eixo primário, desvio padrão no secundário
fig_ida = go.Figure()
fig_ida.add_trace(go.Scatter(
    x=media_ida['ANO'], y=media_ida['mean'],
    mode='lines+markers+text',
    text=media_ida['mean'].round(2),
    textposition='top center',
    name='Média IDA',
    line=dict(color='#2ECC71', width=3),
    marker=dict(size=14)
))
fig_ida.add_trace(go.Bar(
    x=media_ida['ANO'], y=media_ida['std'],
    name='Desvio Padrão',
    marker_color='rgba(46, 204, 113, 0.3)',
    yaxis='y2'
))
fig_ida.update_layout(
    title='📚 Evolução do Desempenho Acadêmico (IDA)',
    xaxis_title='Ano',
    yaxis_title='IDA Médio',
    yaxis2=dict(title='Desvio Padrão', overlaying='y', side='right', showgrid=False),
    template=TEMPLATE,
    legend=dict(x=0.01, y=0.99)
)
fig_ida.show()

print("\n📊 Estatísticas do IDA por ano:")
print(media_ida.round(3).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 11 — IDA médio por classificação (Pedra)
# ─────────────────────────────────────────────────────────────

if 'PEDRA' in df_long.columns:
    ida_pedra = df_long.groupby(['ANO', 'PEDRA'])['IDA'].mean().reset_index()
    ida_pedra = ida_pedra.dropna(subset=['PEDRA'])

    fig_ida_pedra = px.line(
        ida_pedra, x='ANO', y='IDA', color='PEDRA',
        color_discrete_map=CORES_PEDRAS,
        title='📊 IDA Médio por Classificação (Pedra) ao Longo do Tempo',
        markers=True, template=TEMPLATE
    )
    fig_ida_pedra.update_layout(xaxis_title='Ano', yaxis_title='IDA Médio')
    fig_ida_pedra.show()

---
## 6. Análise 3: Engajamento vs Desempenho

O **IEG (Índice de Engajamento)** captura o nível de participação e envolvimento do aluno. 
A hipótese é que **alunos mais engajados têm melhor desempenho** e maior chance de atingir o Ponto de Virada.

**Correlações analisadas:**
- `IEG × IDA` — Engajamento e desempenho acadêmico
- `IEG × IPV` — Engajamento e ponto de virada

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 12 — Análise de Engajamento (IEG) vs Desempenho e IPV
#
# Equivalente à função `analise_engajamento` em src/analysis.py
# ─────────────────────────────────────────────────────────────

df_eng = df_long.dropna(subset=['IEG', 'IDA', 'IPV'])

# IEG × IDA com linha de tendência
fig_ieg_ida = px.scatter(
    df_eng, x='IEG', y='IDA', color='ANO',
    trendline='ols',  # Regressão linear por grupo
    title='🎯 Relação entre Engajamento (IEG) e Desempenho Acadêmico (IDA)',
    labels={'IEG': 'Engajamento (IEG)', 'IDA': 'Desempenho (IDA)'},
    template=TEMPLATE, opacity=0.6
)
fig_ieg_ida.show()

# IEG × IPV
fig_ieg_ipv = px.scatter(
    df_eng, x='IEG', y='IPV', color='ANO',
    trendline='ols',
    title='🔄 Relação entre Engajamento (IEG) e Ponto de Virada (IPV)',
    labels={'IEG': 'Engajamento (IEG)', 'IPV': 'Ponto de Virada (IPV)'},
    template=TEMPLATE, opacity=0.6
)
fig_ieg_ipv.show()

# Correlações
corr_ieg_ida = df_eng[['IEG', 'IDA']].corr().iloc[0, 1]
corr_ieg_ipv = df_eng[['IEG', 'IPV']].corr().iloc[0, 1]

print(f"📈 Correlação IEG × IDA: {corr_ieg_ida:.4f}")
print(f"📈 Correlação IEG × IPV: {corr_ieg_ipv:.4f}")
print("\n💡 Interpretação:")
print("   Correlação > 0.5 indica relação positiva substancial.")
print("   Alunos mais engajados tendem a ter melhor desempenho.")

---
## 7. Análise 4: Autoavaliação (IAA) vs Desempenho Real

O **IAA (Índice de Autoavaliação)** é a percepção que o aluno tem do seu próprio desempenho. A análise verifica se essa autopercepção é **calibrada** com o desempenho real (IDA) ou se há viés (superestimação ou subestimação).

**Métrica chave:** `IAA - IDA`
- **Positivo** → aluno superestima seu desempenho (ponto de atenção)
- **Zero** → percepção calibrada (ideal)
- **Negativo** → aluno subestima suas capacidades

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 13 — Análise de Autoavaliação (IAA) vs Desempenho Real
#
# Equivalente à função `analise_autoavaliacao` em src/analysis.py
# ─────────────────────────────────────────────────────────────

df_iaa = df_long.dropna(subset=['IAA', 'IDA', 'IEG']).copy()
df_iaa['DIFERENCA_IAA_IDA'] = df_iaa['IAA'] - df_iaa['IDA']

# Histograma da diferença IAA - IDA
fig_diff = px.histogram(
    df_iaa, x='DIFERENCA_IAA_IDA', color='ANO',
    nbins=30, barmode='overlay',
    title='📋 Diferença entre Autoavaliação (IAA) e Desempenho Real (IDA)',
    labels={'DIFERENCA_IAA_IDA': 'IAA - IDA', 'count': 'Frequência'},
    template=TEMPLATE, opacity=0.7
)
fig_diff.add_vline(x=0, line_dash='dash', line_color='#6B3FA0',
                   annotation_text='IAA = IDA (percepção calibrada)')
fig_diff.show()

# Scatter IAA x IDA com linha de identidade
fig_scatter = px.scatter(
    df_iaa, x='IAA', y='IDA', color='ANO',
    trendline='ols',
    title='🔍 Coerência: Autoavaliação (IAA) vs Desempenho Real (IDA)',
    template=TEMPLATE, opacity=0.6
)
fig_scatter.add_trace(go.Scatter(
    x=[0, 10], y=[0, 10], mode='lines',
    line=dict(dash='dash', color='rgba(107, 63, 160, 0.4)'),
    name='IAA = IDA'
))
fig_scatter.show()

corr_iaa = df_iaa[['IAA', 'IDA']].corr().iloc[0, 1]
media_diff = df_iaa['DIFERENCA_IAA_IDA'].mean()

print(f"📈 Correlação IAA × IDA: {corr_iaa:.4f}")
print(f"📊 Diferença média (IAA - IDA): {media_diff:.4f}")
print(f"   {'⚠️ Alunos tendem a SUPERESTIMAR seu desempenho' if media_diff > 0.1 else '✅ Autoavaliação bem calibrada'}")

---
## 8. Análise 5: Aspectos Psicossociais (IPS)

O **IPS (Índice Psicossocial)** avalia fatores como bem-estar emocional, condições familiares e suporte social. A hipótese é que alunos com **IPS baixo** apresentam maior risco de queda no desempenho.

**Análises:**
- IPS × IDA (desempenho acadêmico)
- IPS × IEG (engajamento)
- Média de indicadores por grupo de IPS (segmentado em quartis)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 14 — Análise Psicossocial (IPS)
#
# Equivalente à função `analise_psicossocial` em src/analysis.py
# ─────────────────────────────────────────────────────────────

df_ips = df_long.dropna(subset=['IPS', 'IDA', 'IEG'])

# IPS x IDA
fig_ips_ida = px.scatter(
    df_ips, x='IPS', y='IDA', color='ANO', trendline='ols',
    title='🧠 Indicador Psicossocial (IPS) vs Desempenho Acadêmico (IDA)',
    template=TEMPLATE, opacity=0.6
)
fig_ips_ida.show()

# IPS x IEG
fig_ips_ieg = px.scatter(
    df_ips, x='IPS', y='IEG', color='ANO', trendline='ols',
    title='🧠 Indicador Psicossocial (IPS) vs Engajamento (IEG)',
    template=TEMPLATE, opacity=0.6
)
fig_ips_ieg.show()

# Análise por grupo de IPS
df_ips_copy = df_ips.copy()
df_ips_copy['IPS_GRUPO'] = pd.cut(
    df_ips_copy['IPS'], bins=[0, 4, 6, 8, 10],
    labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto']
)

ips_grupo = df_ips_copy.groupby('IPS_GRUPO', observed=True).agg({
    'IDA': 'mean', 'IEG': 'mean', 'INDE': 'mean'
}).reset_index()

fig_grupos = go.Figure()
for col, cor in [('IDA', '#2ECC71'), ('IEG', '#3498DB'), ('INDE', '#E74C3C')]:
    if col in ips_grupo.columns:
        fig_grupos.add_trace(go.Bar(
            x=ips_grupo['IPS_GRUPO'], y=ips_grupo[col],
            name=col, marker_color=cor
        ))
fig_grupos.update_layout(
    title='📊 Desempenho Médio por Grupo de IPS',
    barmode='group', template=TEMPLATE,
    xaxis_title='Grupo IPS', yaxis_title='Média'
)
fig_grupos.show()

corr_ips_ida = df_ips[['IPS', 'IDA']].corr().iloc[0, 1]
corr_ips_ieg = df_ips[['IPS', 'IEG']].corr().iloc[0, 1]
print(f"📈 Correlação IPS × IDA: {corr_ips_ida:.4f}")
print(f"📈 Correlação IPS × IEG: {corr_ips_ieg:.4f}")

---
## 9. Análise 6: Aspectos Psicopedagógicos (IPP)

O **IPP (Índice Psicopedagógico)** é a avaliação realizada pelos psicopedagogos da ONG. A análise verifica se essa avaliação profissional está **alinhada** com o IAN (índice de adequação baseado em dados objetivos).

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 15 — Análise Psicopedagógica (IPP) vs adequação ao nível (IAN)
#
# Equivalente à função `analise_psicopedagogico` em src/analysis.py
# ─────────────────────────────────────────────────────────────

df_ipp = df_long.dropna(subset=['IPP', 'IAN'])

fig_ipp_ian = px.scatter(
    df_ipp, x='IPP', y='IAN', color='ANO', trendline='ols',
    title='📝 Indicador Psicopedagógico (IPP) vs Adequação ao Nível (IAN)',
    template=TEMPLATE, opacity=0.6
)
fig_ipp_ian.show()

# IPP médio por classificação de risco
df_ipp_copy = df_ipp.copy()
df_ipp_copy['RISCO_IAN'] = df_ipp_copy['IAN'].apply(classificar_risco_ian)
ipp_risco = df_ipp_copy.groupby('RISCO_IAN')['IPP'].agg(['mean', 'std']).reset_index()

fig_ipp_risco = px.bar(
    ipp_risco, x='RISCO_IAN', y='mean', error_y='std',
    color='RISCO_IAN', color_discrete_map=CORES_RISCO,
    title='📊 IPP Médio por Classificação de Defasagem (IAN)',
    template=TEMPLATE,
    labels={'mean': 'IPP Médio', 'RISCO_IAN': 'Classificação de Risco'}
)
fig_ipp_risco.show()

corr_ipp_ian = df_ipp[['IPP', 'IAN']].corr().iloc[0, 1]
print(f"📈 Correlação IPP × IAN: {corr_ipp_ian:.4f}")
print("💡 Alta correlação confirma que a avaliação psicopedagógica está alinhada")
print("   com os indicadores objetivos de defasagem.")

---
## 10. Análise 7: Ponto de Virada (IPV)

O **Ponto de Virada** é um dos conceitos mais importantes no programa Passos Mágicos — representa o momento transformador na trajetória do aluno. O **IPV** mede a progressão rumo a esse ponto.

**Análises:**
- Quais indicadores têm maior correlação com o IPV?
- Como se diferenciam os alunos que **atingiram** vs **não atingiram** o Ponto de Virada?

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 16 — Análise do Ponto de Virada (IPV)
#
# Equivalente à função `analise_ponto_virada` em src/analysis.py
# ─────────────────────────────────────────────────────────────

df_ipv = df_long.dropna(subset=['IPV', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP'])

# Correlações de cada indicador com o IPV
indicadores = ['IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IAN']
correlacoes = {ind: df_ipv[['IPV', ind]].corr().iloc[0, 1]
               for ind in indicadores if ind in df_ipv.columns}

corr_df = pd.DataFrame({
    'Indicador': list(correlacoes.keys()),
    'Correlação com IPV': list(correlacoes.values())
}).sort_values('Correlação com IPV', ascending=True)

fig_corr = px.bar(
    corr_df, x='Correlação com IPV', y='Indicador', orientation='h',
    color='Correlação com IPV', color_continuous_scale='RdYlGn',
    title='🏆 Correlação dos Indicadores com o Ponto de Virada (IPV)',
    template=TEMPLATE
)
fig_corr.show()

# Comparação entre alunos que atingiram vs não atingiram o PV
if 'PONTO_VIRADA' in df_ipv.columns:
    pv_comp = df_ipv.groupby('PONTO_VIRADA')[indicadores].mean().T.reset_index()
    pv_comp.columns = ['Indicador'] + list(pv_comp.columns[1:])

    if 'Sim' in pv_comp.columns and 'Não' in pv_comp.columns:
        fig_pv = go.Figure()
        fig_pv.add_trace(go.Bar(
            x=pv_comp['Indicador'], y=pv_comp['Não'],
            name='Não atingiu PV', marker_color='#E74C3C'
        ))
        fig_pv.add_trace(go.Bar(
            x=pv_comp['Indicador'], y=pv_comp['Sim'],
            name='Atingiu PV', marker_color='#2ECC71'
        ))
        fig_pv.update_layout(
            title='🔄 Indicadores: Alunos que Atingiram vs Não Atingiram o Ponto de Virada',
            barmode='group', template=TEMPLATE,
            xaxis_title='Indicador', yaxis_title='Média'
        )
        fig_pv.show()

print("\n📊 Correlações dos indicadores com IPV:")
for ind, corr in sorted(correlacoes.items(), key=lambda x: x[1], reverse=True):
    print(f"   {ind}: {corr:.4f}")

---
## 11. Análise 8: Multidimensionalidade dos Indicadores

O **INDE** é um índice composto. Esta análise explora quais indicadores têm maior peso na formação do INDE e como eles se inter-relacionam, usando:
- **Matriz de correlação** entre todos os indicadores
- **Gráfico de barras** da correlação de cada indicador com o INDE
- **Radar chart** do perfil médio por classificação (pedra)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 17 — Análise Multidimensional dos Indicadores
#
# Equivalente à função `analise_multidimensional` em src/analysis.py
# ─────────────────────────────────────────────────────────────

indicadores_all = ['IDA', 'IEG', 'IPS', 'IPP', 'IAA', 'IPV', 'IAN']
df_multi = df_long.dropna(subset=indicadores_all + ['INDE'])

# Matriz de correlação
corr_matrix = df_multi[indicadores_all + ['INDE']].corr()

fig_heatmap = px.imshow(
    corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    color_continuous_scale='RdBu_r',
    title='🔗 Matriz de Correlação entre Todos os Indicadores e o INDE',
    template=TEMPLATE,
    text_auto='.2f',
    aspect='auto'
)
fig_heatmap.show()

# Importância dos indicadores para o INDE
corr_inde = corr_matrix['INDE'].drop('INDE').sort_values(ascending=True)

fig_import = px.bar(
    x=corr_inde.values, y=corr_inde.index, orientation='h',
    color=corr_inde.values, color_continuous_scale='Viridis',
    title='📊 Correlação de Cada Indicador com o INDE',
    labels={'x': 'Correlação', 'y': 'Indicador'},
    template=TEMPLATE
)
fig_import.show()

print("📊 Correlação de cada indicador com o INDE:")
for ind, val in corr_inde.sort_values(ascending=False).items():
    print(f"   {ind}: {val:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 18 — Radar Chart: perfil dos indicadores por pedra
#
# Visualização polar que permite comparar o perfil médio de cada
# categoria (Quartzo, Ágata, Ametista, Topázio) em todos os eixos
# simultaneamente.
# ─────────────────────────────────────────────────────────────

if 'PEDRA' in df_multi.columns:
    pedra_media = df_multi.groupby('PEDRA')[indicadores_all].mean()
    pedra_media = pedra_media.reindex(['Quartzo', 'Ágata', 'Ametista', 'Topázio'])
    pedra_media = pedra_media.dropna()

    fig_radar = go.Figure()
    for pedra in pedra_media.index:
        valores = pedra_media.loc[pedra].tolist()
        valores.append(valores[0])  # Fechar o polígono
        categorias = indicadores_all + [indicadores_all[0]]
        fig_radar.add_trace(go.Scatterpolar(
            r=valores, theta=categorias,
            fill='toself', name=pedra,
            line_color=CORES_PEDRAS.get(pedra, '#95A5A6')
        ))
    fig_radar.update_layout(
        title='🎯 Radar: Perfil dos Indicadores por Classificação (Pedra)',
        polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
        template=TEMPLATE
    )
    fig_radar.show()

---
## 12. Análise 9: Efetividade do Programa

Analisamos se a Passos Mágicos está de fato promovendo **melhora consistente** nos alunos ao longo dos anos:
- Evolução do INDE médio por classificação (pedra)
- Distribuição dos alunos por pedra ao longo dos anos (migração para categorias superiores?)
- Taxa de alunos que atingiram o Ponto de Virada por ano

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 19 — Análise de Efetividade do Programa
#
# Equivalente à função `analise_efetividade` em src/analysis.py
# ─────────────────────────────────────────────────────────────

# ── Evolução do INDE por pedra ──
if 'PEDRA' in df_long.columns:
    pedra_ano = df_long.groupby(['ANO', 'PEDRA']).agg(
        INDE=('INDE', 'mean'), IDA=('IDA', 'mean'), IEG=('IEG', 'mean')
    ).reset_index().dropna(subset=['PEDRA'])

    fig_inde_evo = px.line(
        pedra_ano, x='ANO', y='INDE', color='PEDRA',
        color_discrete_map=CORES_PEDRAS, markers=True,
        title='📈 Evolução do INDE Médio por Classificação (Pedra)',
        template=TEMPLATE
    )
    fig_inde_evo.show()

# ── Distribuição de pedras por ano (stacked bar) ──
pedra_dist = df_long.groupby(['ANO', 'PEDRA']).size().reset_index(name='N')
pedra_dist = pedra_dist.dropna(subset=['PEDRA'])
total_ano = df_long.groupby('ANO').size().reset_index(name='TOTAL')
pedra_dist = pedra_dist.merge(total_ano, on='ANO')
pedra_dist['PERCENTUAL'] = (pedra_dist['N'] / pedra_dist['TOTAL'] * 100).round(1)

fig_pedra_dist = px.bar(
    pedra_dist, x='ANO', y='PERCENTUAL', color='PEDRA',
    color_discrete_map=CORES_PEDRAS,
    title='📊 Distribuição dos Alunos por Classificação (Pedra) ao Longo dos Anos',
    barmode='stack', template=TEMPLATE
)
fig_pedra_dist.show()

# ── Taxa de Ponto de Virada por ano ──
if 'PONTO_VIRADA' in df_long.columns:
    pv_ano = df_long.groupby('ANO')['PONTO_VIRADA'].apply(
        lambda x: (x == 'Sim').sum() / len(x) * 100
    ).reset_index(name='TAXA_PV')

    fig_taxa_pv = px.bar(
        pv_ano, x='ANO', y='TAXA_PV',
        title='🏆 Taxa de Ponto de Virada (%) por Ano',
        color='TAXA_PV', color_continuous_scale='Greens',
        template=TEMPLATE, text='TAXA_PV'
    )
    fig_taxa_pv.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig_taxa_pv.update_layout(yaxis_title='% de Alunos que Atingiram PV')
    fig_taxa_pv.show()

    print("📊 Taxa de Ponto de Virada por ano:")
    print(pv_ano.to_string(index=False))

---
## 13. Análise 10: Insights Adicionais

Análises complementares que enriquecem a compreensão do perfil dos alunos:
- Desempenho médio **por gênero**
- Impacto do **tempo de permanência** na ONG (`ANOS_PM`)
- Distribuição de **idades**
- Indicadores **por fase**
- Diferença entre alunos **indicados vs não indicados para bolsa**

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 20 — Insights Adicionais
#
# Equivalente à função `analise_insights` em src/analysis.py
# ─────────────────────────────────────────────────────────────

# ── Por Gênero ──
if 'GENERO' in df_xlsx.columns:
    cols_gen = ['INDE', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV']
    cols_gen_avail = [c for c in cols_gen if c in df_xlsx.columns]
    genero = df_xlsx.groupby('GENERO')[cols_gen_avail].mean().T.reset_index()
    genero.columns = ['Indicador'] + list(genero.columns[1:])

    fig_genero = go.Figure()
    cores_gen = {'Menina': '#FF69B4', 'Menino': '#4169E1'}
    for g in ['Menina', 'Menino']:
        if g in genero.columns:
            fig_genero.add_trace(go.Bar(
                x=genero['Indicador'], y=genero[g],
                name=g, marker_color=cores_gen[g]
            ))
    fig_genero.update_layout(
        title='👫 Indicadores Médios por Gênero',
        barmode='group', template=TEMPLATE
    )
    fig_genero.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 21 — Impacto do tempo na PM e análise por fase
# ─────────────────────────────────────────────────────────────

# Tempo na PM vs indicadores
if 'ANOS_PM' in df_xlsx.columns:
    cols_pm = [c for c in ['INDE', 'IDA', 'IEG'] if c in df_xlsx.columns]
    anos_pm = df_xlsx.groupby('ANOS_PM')[cols_pm].mean().reset_index()
    fig_anos_pm = px.line(
        anos_pm, x='ANOS_PM', y=cols_pm, markers=True,
        title='⏰ Indicadores Médios por Tempo na Passos Mágicos (anos)',
        labels={'ANOS_PM': 'Anos na PM', 'value': 'Média', 'variable': 'Indicador'},
        template=TEMPLATE
    )
    fig_anos_pm.show()

# Distribuição de idades
if 'IDADE' in df_xlsx.columns:
    fig_idade = px.histogram(
        df_xlsx, x='IDADE',
        color='GENERO' if 'GENERO' in df_xlsx.columns else None,
        nbins=15, barmode='overlay', opacity=0.7,
        title='📊 Distribuição de Idades dos Alunos',
        color_discrete_map={'Menina': '#FF69B4', 'Menino': '#4169E1'},
        template=TEMPLATE
    )
    fig_idade.show()

# Indicadores por fase
if 'FASE' in df_xlsx.columns:
    cols_fase = [c for c in ['INDE', 'IDA', 'IEG', 'IAA', 'IPS'] if c in df_xlsx.columns]
    fase_stats = df_xlsx.groupby('FASE')[cols_fase].mean().reset_index()
    fig_fase = px.line(
        fase_stats, x='FASE', y=cols_fase, markers=True,
        title='📚 Indicadores Médios por Fase do Programa',
        labels={'FASE': 'Fase', 'value': 'Média', 'variable': 'Indicador'},
        template=TEMPLATE
    )
    fig_fase.show()

# Indicados para bolsa
if 'INDICADO_BOLSA' in df_xlsx.columns:
    cols_bolsa = [c for c in ['INDE', 'IDA', 'IEG'] if c in df_xlsx.columns]
    bolsa_stats = df_xlsx.groupby('INDICADO_BOLSA')[cols_bolsa].mean().reset_index()
    fig_bolsa = px.bar(
        bolsa_stats, x='INDICADO_BOLSA', y=cols_bolsa,
        barmode='group',
        title='🎓 Indicadores: Indicados vs Não Indicados para Bolsa',
        template=TEMPLATE
    )
    fig_bolsa.show()

---
## 14. Preparação dos Dados para Modelagem

### 🎯 Definição do Target
**RISCO** = 1 se `DEFASAGEM < 0` (aluno atrás do nível esperado para sua fase), 0 caso contrário.

### ⚠️ Decisões Críticas Anti-Data-Leakage

| Feature Removida | Motivo |
|-----------------|--------|
| `NOTA_MAT / NOTA_PORT` | Correlação 0.97 com `IDA` — redundância pura |
| `IDADE` | Define `FASE_IDEAL` → determina `DEFASAGEM` → determina `RISCO`. Modelo trivial! |
| `NUM_AVALIADORES` | Proxy da FASE, sem valor preditivo independente |
| `IAN / INDE` | Codificam diretamente a defasagem — leakagem severa |

### ✅ Features Selecionadas (9 variáveis)

Apenas indicadores que um educador pode observar **antes** de saber o nível de defasagem do aluno:

| Feature | Descrição |
|---------|----------|
| `IAA` | Autoavaliação do aluno |
| `IEG` | Engajamento no programa |
| `IPS` | Indicador psicossocial |
| `IDA` | Desempenho acadêmico |
| `IPV` | Ponto de virada |
| `FASE` | Fase atual (nível) do aluno |
| `ANOS_PM` | Tempo na associação (engajamento acumulado) |
| `GENERO_NUM` | Gênero codificado (0=Menina, 1=Menino) |
| `PV_NUM` | Atingiu ponto de virada? (0/1) |

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 22 — Preparação do dataset para modelagem
#
# Equivalente à função `criar_dataset_modelo` em src/data_loader.py
# ─────────────────────────────────────────────────────────────

def criar_dataset_modelo(df_xlsx):
    """
    Prepara o dataset para modelagem preditiva de risco de defasagem.

    Target: RISCO = 1 se DEFASAGEM < 0 (aluno abaixo do nível esperado)
    """
    df = df_xlsx.copy()

    # ── Criar target binário ──
    df['RISCO'] = (df['DEFASAGEM'] < 0).astype(int)

    # ── Encoding de variáveis categóricas ──
    # Menino=1, Menina=0 (codificação binária direta)
    df['GENERO_NUM'] = (df['GENERO'] == 'Menino').astype(int)
    # Atingiu PV? Sim=1, Não=0
    df['PV_NUM'] = (df['PONTO_VIRADA'] == 'Sim').astype(int)

    # ── Features legítimas (sem data leakage) ──
    features = [
        'IAA',        # Autoavaliação
        'IEG',        # Engajamento
        'IPS',        # Psicossocial
        'IDA',        # Desempenho acadêmico
        'IPV',        # Ponto de virada
        'FASE',       # Fase/nível atual
        'ANOS_PM',    # Tempo na associação
        'GENERO_NUM',
        'PV_NUM',
    ]

    # Remover linhas com valores faltantes nas features essenciais
    df_modelo = df.dropna(subset=['IAA', 'IEG', 'IPS', 'IDA', 'IPV', 'FASE', 'ANOS_PM'])

    return df_modelo, features


df_modelo, features = criar_dataset_modelo(df_xlsx)

print("✅ Dataset para modelagem preparado!")
print(f"   Amostras totais: {len(df_modelo)}")
print(f"   Features: {features}")
print(f"\n📊 Distribuição do target RISCO:")
risco_dist = df_modelo['RISCO'].value_counts()
print(f"   Sem risco (0): {risco_dist.get(0, 0)} ({risco_dist.get(0, 0)/len(df_modelo)*100:.1f}%)")
print(f"   Em risco  (1): {risco_dist.get(1, 0)} ({risco_dist.get(1, 0)/len(df_modelo)*100:.1f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 23 — Visualização do balanceamento do dataset
# ─────────────────────────────────────────────────────────────

# Distribuição do target
fig_target = px.pie(
    values=[risco_dist.get(0, 0), risco_dist.get(1, 0)],
    names=['Sem Risco (0)', 'Em Risco (1)'],
    title='🎯 Distribuição do Target: Risco de Defasagem',
    color_discrete_sequence=['#2ECC71', '#E74C3C'],
    template=TEMPLATE, hole=0.4
)
fig_target.show()

# Análise descritiva das features por classe
print("\n📊 Médias das features por classe (RISCO):")
print(df_modelo.groupby('RISCO')[features].mean().round(3).to_string())

---
## 15. Treinamento e Avaliação dos Modelos

### 🔬 Estratégia de Modelagem

Treinamos **4 modelos** diferentes para encontrar o melhor para este problema:

| Modelo | Características |
|--------|----------------|
| **Random Forest** | Ensemble de árvores, robusto a overfiting, `class_weight='balanced'` |
| **XGBoost** | Gradient boosting otimizado, `scale_pos_weight` para classes desbalanceadas |
| **Gradient Boosting** | Boosting sequencial com regularização |
| **Logistic Regression** | Modelo linear baseline, `C=0.1` (regularização forte) |

### 📏 Métricas de Avaliação
- **Accuracy** — % de predições corretas
- **Precision** — dos que foram preditos como "em risco", quantos realmente são?
- **Recall** — dos que estão realmente em risco, quantos o modelo pegou?
- **F1-Score** — harmônica de precision e recall (principal critério de seleção)
- **ROC-AUC** — capacidade de discriminação entre as classes
- **Cross-validation F1** — generalização com 5-fold estratificado

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 24 — Split treino/teste e normalização
#
# Equivalente ao método `preparar_dados` da classe ModeloRiscoDefasagem
# em src/model.py
# ─────────────────────────────────────────────────────────────

X = df_modelo[features].copy()
y = df_modelo['RISCO'].copy()

# Imputar medianas nos raros valores nulos remanescentes
X = X.fillna(X.median())

# Split estratificado: mantém proporção das classes em treino e teste
# test_size=0.2 → 80% treino, 20% teste
# stratify=y    → garante mesma proporção de RISCO em ambos os splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler: normaliza para média=0 e std=1
# Necessário apenas para Logistic Regression (influenciada pela escala)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit+transform no treino
X_test_scaled  = scaler.transform(X_test)         # apenas transform no teste (sem data leakage)

print("✅ Dados divididos e normalizados!")
print(f"   Treino: {X_train.shape[0]} amostras")
print(f"   Teste:  {X_test.shape[0]} amostras")
print(f"   Balanceamento (treino) → Sem risco: {(y_train==0).sum()} | Em risco: {(y_train==1).sum()}")
print(f"   Balanceamento (teste)  → Sem risco: {(y_test==0).sum()} | Em risco: {(y_test==1).sum()}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 25 — Configuração e treinamento dos 4 modelos
#
# Equivalente ao método `treinar_modelos` da classe ModeloRiscoDefasagem
# em src/model.py
#
# Hiperparâmetros conservadores para dataset pequeno (~860 linhas):
#   • max_depth baixo → menos overfitting
#   • min_samples_leaf alto → folhas mais representativas
#   • class_weight='balanced' → penaliza mais erros na classe minoritária
# ─────────────────────────────────────────────────────────────

modelos_config = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=5,
        min_samples_split=10, min_samples_leaf=5,
        max_features='sqrt', random_state=42,
        class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.05,
        min_child_weight=5, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=1.0, reg_lambda=2.0, random_state=42,
        eval_metric='logloss',
        # scale_pos_weight: balanceia o peso das classes no XGBoost
        scale_pos_weight=len(y_train[y_train==0]) / max(len(y_train[y_train==1]), 1)
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.05,
        min_samples_split=10, min_samples_leaf=5,
        subsample=0.8, random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        C=0.1,          # C baixo = regularização L2 forte
        max_iter=1000, random_state=42,
        class_weight='balanced'
    )
}

# ── Treinamento e avaliação ──
resultados = {}
melhor_f1 = 0
melhor_nome = None
melhor_modelo = None

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nome, modelo in modelos_config.items():
    print(f"\n{'='*55}")
    print(f"   Treinando: {nome}")
    print(f"{'='*55}")

    # Logistic Regression usa dados normalizados; outros usam dados brutos
    if nome == 'Logistic Regression':
        modelo.fit(X_train_scaled, y_train)
        y_pred  = modelo.predict(X_test_scaled)
        y_proba = modelo.predict_proba(X_test_scaled)[:, 1]
        cv_scores = cross_val_score(modelo, X_train_scaled, y_train, cv=cv, scoring='f1')
    else:
        modelo.fit(X_train, y_train)
        y_pred  = modelo.predict(X_test)
        y_proba = modelo.predict_proba(X_test)[:, 1]
        cv_scores = cross_val_score(modelo, X_train, y_train, cv=cv, scoring='f1')

    # Calcular métricas de avaliação
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    roc  = roc_auc_score(y_test, y_proba)
    cm   = confusion_matrix(y_test, y_pred)

    resultados[nome] = {
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1_score': f1, 'roc_auc': roc,
        'confusion_matrix': cm,
        'y_pred': y_pred, 'y_proba': y_proba,
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
        'classification_report': classification_report(y_test, y_pred, output_dict=True)
    }

    print(f"   Accuracy:  {acc:.4f}")
    print(f"   Precision: {prec:.4f}")
    print(f"   Recall:    {rec:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    print(f"   ROC-AUC:   {roc:.4f}")
    print(f"   CV F1:     {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

    if f1 > melhor_f1:
        melhor_f1 = f1
        melhor_nome = nome
        melhor_modelo = modelo

print(f"\n{'='*55}")
print(f"   🏆 MELHOR MODELO: {melhor_nome} (F1: {melhor_f1:.4f})")
print(f"{'='*55}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 26 — Comparação visual dos modelos
#
# Equivalente ao método `obter_comparacao_modelos` da classe
# ModeloRiscoDefasagem em src/model.py
# ─────────────────────────────────────────────────────────────

comparacao = []
for nome, res in resultados.items():
    comparacao.append({
        'Modelo': nome,
        'Accuracy': res['accuracy'],
        'Precision': res['precision'],
        'Recall': res['recall'],
        'F1-Score': res['f1_score'],
        'ROC-AUC': res['roc_auc'],
        'CV F1 (média)': res['cv_f1_mean'],
        'CV F1 (std)': res['cv_f1_std']
    })

df_comp = pd.DataFrame(comparacao).sort_values('F1-Score', ascending=False)
print("📊 Comparação dos Modelos:")
print(df_comp.round(4).to_string(index=False))

# Gráfico comparativo
fig_comp = px.bar(
    df_comp.melt(id_vars='Modelo', value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']),
    x='Modelo', y='value', color='variable', barmode='group',
    title='📊 Comparação de Métricas entre os Modelos de ML',
    labels={'value': 'Pontuação', 'variable': 'Métrica'},
    template=TEMPLATE
)
fig_comp.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 27 — Curva ROC e Feature Importance do melhor modelo
# ─────────────────────────────────────────────────────────────

# Curva ROC de todos os modelos
fig_roc = go.Figure()
cores_modelos = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

for (nome, res), cor in zip(resultados.items(), cores_modelos):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines', name=f"{nome} (AUC={res['roc_auc']:.3f})",
        line=dict(color=cor, width=2)
    ))

fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(dash='dash', color='gray'), name='Random (AUC=0.5)'
))
fig_roc.update_layout(
    title='📈 Curvas ROC — Comparação dos Modelos',
    xaxis_title='Taxa de Falso Positivo (FPR)',
    yaxis_title='Taxa de Verdadeiro Positivo (TPR / Recall)',
    template=TEMPLATE
)
fig_roc.show()

# Feature importance do melhor modelo (se disponível)
if hasattr(melhor_modelo, 'feature_importances_'):
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': melhor_modelo.feature_importances_
    }).sort_values('importance', ascending=True)

    fig_feat = px.bar(
        feat_imp, x='importance', y='feature', orientation='h',
        title=f'🔑 Importância das Features — {melhor_nome}',
        color='importance', color_continuous_scale='Viridis',
        template=TEMPLATE,
        labels={'importance': 'Importância', 'feature': 'Feature'}
    )
    fig_feat.show()

    print(f"\n🔑 Feature Importance ({melhor_nome}):")
    for _, row in feat_imp.sort_values('importance', ascending=False).iterrows():
        print(f"   {row['feature']:15s}: {row['importance']:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 28 — Matriz de Confusão do melhor modelo
# ─────────────────────────────────────────────────────────────

cm_melhor = resultados[melhor_nome]['confusion_matrix']
TN, FP, FN, TP = cm_melhor.ravel()

fig_cm = px.imshow(
    cm_melhor,
    x=['Predito: Sem Risco', 'Predito: Em Risco'],
    y=['Real: Sem Risco', 'Real: Em Risco'],
    title=f'🔍 Matriz de Confusão — {melhor_nome}',
    color_continuous_scale='Blues',
    text_auto=True, template=TEMPLATE
)
fig_cm.show()

print(f"\n📋 Interpretação da Matriz de Confusão:")
print(f"   ✅ Verdadeiros Negativos (TN): {TN}  — Sem risco corretamente identificados")
print(f"   ❌ Falsos Positivos (FP):     {FP}  — Sem risco erroneamente alertados")
print(f"   ⚠️  Falsos Negativos (FN):     {FN}  — Em risco não detectados (mais crítico!)")
print(f"   ✅ Verdadeiros Positivos (TP): {TP}  — Em risco corretamente detectados")
print(f"\n   Recall (sensibilidade): {TP / (TP + FN):.1%}  — % dos alunos em risco que o modelo detecta")

print(f"\n📋 Classification Report completo ({melhor_nome}):")
print(classification_report(y_test, resultados[melhor_nome]['y_pred'],
                             target_names=['Sem Risco', 'Em Risco']))

---
## 16. Predição Individual de Risco

O modelo treinado pode ser usado para **predizer o risco de um aluno específico** em tempo real, dado o conjunto de indicadores observados. Esta funcionalidade alimenta a aplicação Streamlit (`app.py`).

### Como funciona:
1. Recebe um dicionário com as features do aluno
2. Estrutura como DataFrame
3. Aplica normalização (se Logistic Regression)
4. Retorna probabilidade e classificação

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 29 — Função de predição individual
#
# Equivalente ao método `predizer_risco` da classe ModeloRiscoDefasagem
# em src/model.py
# ─────────────────────────────────────────────────────────────

def predizer_risco(dados_aluno, modelo, features_list, scaler_obj, nome_modelo):
    """
    Realiza predição de risco de defasagem para um único aluno.

    Args:
        dados_aluno   : dict com indicadores do aluno
        modelo        : modelo treinado
        features_list : lista de features usadas no treinamento
        scaler_obj    : StandardScaler treinado (para Logistic Regression)
        nome_modelo   : nome do melhor modelo

    Returns:
        dict com 'risco', 'probabilidade_sem_risco',
                 'probabilidade_risco', 'classificacao'
    """
    df_aluno = pd.DataFrame([dados_aluno])

    # Garantir que todas as features estão presentes
    for f in features_list:
        if f not in df_aluno.columns:
            df_aluno[f] = 0  # preencher ausentes com 0

    df_aluno = df_aluno[features_list]

    if nome_modelo == 'Logistic Regression':
        df_scaled = scaler_obj.transform(df_aluno)
        probabilidade = modelo.predict_proba(df_scaled)[0]
        predicao = modelo.predict(df_scaled)[0]
    else:
        probabilidade = modelo.predict_proba(df_aluno)[0]
        predicao = modelo.predict(df_aluno)[0]

    return {
        'risco': int(predicao),
        'probabilidade_sem_risco': float(probabilidade[0]),
        'probabilidade_risco': float(probabilidade[1]),
        'classificacao': 'EM RISCO' if predicao == 1 else 'SEM RISCO'
    }


# ── Exemplo 1: Aluno com bons indicadores ──
aluno_bom = {
    'IAA': 8.5, 'IEG': 8.0, 'IPS': 7.5,
    'IDA': 8.0, 'IPV': 9.0, 'FASE': 5,
    'ANOS_PM': 4, 'GENERO_NUM': 0, 'PV_NUM': 1
}

# ── Exemplo 2: Aluno com indicadores de alerta ──
aluno_risco = {
    'IAA': 5.0, 'IEG': 4.5, 'IPS': 4.0,
    'IDA': 4.5, 'IPV': 5.0, 'FASE': 3,
    'ANOS_PM': 1, 'GENERO_NUM': 1, 'PV_NUM': 0
}

res_bom   = predizer_risco(aluno_bom,   melhor_modelo, features, scaler, melhor_nome)
res_risco = predizer_risco(aluno_risco, melhor_modelo, features, scaler, melhor_nome)

print("🧪 TESTE DE PREDIÇÃO")
print("─" * 50)
print(f"\n👨‍🎓 Aluno com bons indicadores:")
print(f"   Classificação: {res_bom['classificacao']}")
print(f"   Probabilidade de risco: {res_bom['probabilidade_risco']:.1%}")

print(f"\n⚠️  Aluno com indicadores de alerta:")
print(f"   Classificação: {res_risco['classificacao']}")
print(f"   Probabilidade de risco: {res_risco['probabilidade_risco']:.1%}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 30 — Visualização da predição
# ─────────────────────────────────────────────────────────────

fig_pred = go.Figure()

for label, res, cor in [
    ('Aluno com Bons Indicadores', res_bom, '#2ECC71'),
    ('Aluno em Alerta', res_risco, '#E74C3C')
]:
    fig_pred.add_trace(go.Bar(
        name=label,
        x=['Sem Risco', 'Em Risco'],
        y=[res['probabilidade_sem_risco'], res['probabilidade_risco']],
        marker_color=cor,
        text=[f"{res['probabilidade_sem_risco']:.1%}", f"{res['probabilidade_risco']:.1%}"],
        textposition='outside'
    ))

fig_pred.update_layout(
    title='🎯 Probabilidades de Risco — Comparação entre Perfis de Alunos',
    barmode='group', template=TEMPLATE,
    yaxis_title='Probabilidade', yaxis_range=[0, 1.1]
)
fig_pred.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 31 — Salvar o modelo treinado
#
# Equivalente ao método `salvar_modelo` da classe ModeloRiscoDefasagem
# em src/model.py
#
# O arquivo .joblib contém:
#   • modelo: o estimador treinado
#   • scaler: o StandardScaler ajustado
#   • features: lista de features usadas
#   • melhor_nome: nome do algoritmo vencedor
#   • resultados: dicionário com métricas de todos os modelos
# ─────────────────────────────────────────────────────────────

artefatos = {
    'modelo': melhor_modelo,
    'scaler': scaler,
    'features': features,
    'melhor_nome': melhor_nome,
    'resultados': {
        nome: {
            'accuracy': res['accuracy'],
            'precision': res['precision'],
            'recall': res['recall'],
            'f1_score': res['f1_score'],
            'roc_auc': res['roc_auc'],
        }
        for nome, res in resultados.items()
    }
}

caminho_modelo = '../models/modelo_risco_notebook.joblib'
joblib.dump(artefatos, caminho_modelo)
print(f"✅ Modelo salvo em: {caminho_modelo}")
print(f"   Algoritmo: {melhor_nome}")
print(f"   F1-Score final: {resultados[melhor_nome]['f1_score']:.4f}")

---
## 19. Modelo 3: Classificação de Melhor Matéria

### 🎓 Objetivo

Prever em qual matéria cada aluno se destaca mais (Matemática, Português ou Inglês)
usando **apenas indicadores educacionais** — sem incluir as notas brutas como feature.

### 🔑 Anti-leakage

| Elemento | Decisão |
|---|---|
| **Target** | Matéria com maior nota (derivada de NOTA_MAT/PORT/ING) |
| **Features** | IAA, IEG, IPS, IDA, IPV, FASE — nunca as notas brutas |
| **Proibido** | Usar NOTA_MAT/PORT/ING como feature (seria leakage direto) |
| **Algoritmo** | Random Forest multiclasse com class_weight='balanced' |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 38 — Construção do Dataset de Melhor Matéria
# ─────────────────────────────────────────────────────────────────────────

from train_model import criar_dataset_materias, FEAT_MATERIAS

df_mat = criar_dataset_materias(df_xlsx)

dist = df_mat["Melhor_Materia"].value_counts()
print("=" * 55)
print("  Dataset — Melhor Matéria")
print("=" * 55)
print(f"  Alunos com notas: {len(df_mat)}")
for mat, cnt in dist.items():
    print(f"  {mat}: {cnt} ({cnt/len(df_mat)*100:.0f}%)")
print(f"  Features ({len(FEAT_MATERIAS)}): {FEAT_MATERIAS}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 39 — Visualização: Distribuição e Nota Média por Matéria
# ─────────────────────────────────────────────────────────────────────────

import plotly.express as px

fig1 = px.pie(
    df_mat, names='Melhor_Materia',
    color='Melhor_Materia',
    color_discrete_map={'Matemática':'#4CAF50','Português':'#2196F3','Inglês':'#FF9800'},
    title='🏆 Distribuição da Melhor Matéria',
    template='plotly_white', hole=0.4
)
fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.show()

medias = pd.DataFrame({
    'Matéria': ['Matemática','Português','Inglês'],
    'Média': [df_mat['NOTA_MAT'].mean(), df_mat['NOTA_PORT'].mean(), df_mat['NOTA_ING'].mean()]
}).dropna()
fig2 = px.bar(
    medias, x='Matéria', y='Média',
    color='Matéria',
    color_discrete_map={'Matemática':'#4CAF50','Português':'#2196F3','Inglês':'#FF9800'},
    title='📊 Nota Média por Matéria',
    template='plotly_white', text_auto='.2f'
)
fig2.update_layout(showlegend=False)
fig2.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 40 — Treinamento do Modelo de Melhor Matéria
# Usa treinar_materias() de train_model.py (arquivo autônomo)
# ─────────────────────────────────────────────────────────────────────────

from train_model import treinar_materias

print("Treinando Random Forest multiclasse...")
art_materias = treinar_materias()
res = art_materias["resultados"]
print("=" * 55)
print("  Resultados — 5-Fold Stratified CV")
print("=" * 55)
print(f"  Acurácia   : {res["cv_acc_mean"]*100:.1f}% ± {res["cv_acc_std"]*100:.1f}%")
print(f"  F1 Weighted: {res["cv_f1_mean"]*100:.1f}% ± {res["cv_f1_std"]*100:.1f}%")
print(f"  Train Acc  : {res["train_acc"]*100:.1f}%")
print(f"  Gap Overfit: {res["overfitting_gap"]:.3f}")
print(f"  Classes    : {res["classes"]}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 41 — Feature Importance e CV por Fold
# ─────────────────────────────────────────────────────────────────────────

fig_imp = px.bar(
    modelo_materias.feature_importance,
    x='Importância', y='Feature', orientation='h',
    color='Importância', color_continuous_scale='Blues',
    title='🎯 Importância das Features — Melhor Matéria',
    template='plotly_white'
)
fig_imp.update_layout(coloraxis_showscale=False)
fig_imp.show()

cv_df = pd.DataFrame({
    'Fold':     [f'Fold {i+1}' for i in range(5)],
    'Acurácia': res['cv_acc_scores'],
})
fig_cv = px.bar(
    cv_df, x='Fold', y='Acurácia',
    color_discrete_sequence=['#2D325E'],
    title='📈 Acurácia por Fold — 5-Fold CV (Melhor Matéria)',
    template='plotly_white', text_auto='.1%'
)
fig_cv.add_hline(y=res['cv_acc_mean'], line_dash='dash', line_color='#EE8133',
                 annotation_text=f"Média: {res['cv_acc_mean']:.1%}")
fig_cv.show()

top = modelo_materias.feature_importance.iloc[-1]['Feature']
print(f'💡 Feature mais importante: {top}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 42 — Predição Individual
# ─────────────────────────────────────────────────────────────────────────

# Aluno com perfil de destaque em Matemática
aluno_mat = {'IAA':7.5,'IEG':8.0,'IPS':7.0,'IDA':8.5,'IPV':7.5,'FASE':3}
# Aluno com perfil de destaque em Português
aluno_port = {'IAA':8.5,'IEG':7.0,'IPS':8.0,'IDA':7.5,'IPV':8.0,'FASE':2}

pred_mat  = modelo_materias.predizer_melhor_materia(aluno_mat)
pred_port = modelo_materias.predizer_melhor_materia(aluno_port)

print('─' * 50)
print(f"Perfil 1 — Melhor matéria prevista: {pred_mat['materia']}")
print(f"  Confiança: {pred_mat['confianca']*100:.1f}%")
for mat, prob in pred_mat['probabilidades'].items():
    print(f"  {mat}: {prob*100:.1f}%")
print()
print(f"Perfil 2 — Melhor matéria prevista: {pred_port['materia']}")
print(f"  Confiança: {pred_port['confianca']*100:.1f}%")
for mat, prob in pred_port['probabilidades'].items():
    print(f"  {mat}: {prob*100:.1f}%")
print('─' * 50)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 43 — Verificação do Modelo de Melhor Matéria Salvo
# O modelo já foi salvo por treinar_materias() na célula anterior.
# ─────────────────────────────────────────────────────────────────────────

import os, joblib

print("Modelos disponíveis em models/:")
for f in sorted(os.listdir("models")):
    size = os.path.getsize(f"models/{f}") / 1024
    print(f"   {f}  ({size:.0f} KB)")

# Validação: carregar e testar a predição
art_val = joblib.load("models/modelo_materias.joblib")
modelo_val = art_val["modelo"]
le_val     = art_val["label_encoder"]
feats_val  = art_val["features"]

aluno_teste = {"IAA": 7.5, "IEG": 8.0, "IPS": 7.0, "IDA": 8.5, "IPV": 7.5, "FASE": 3}
import numpy as np
X_val = np.array([[aluno_teste[f] for f in feats_val]])
prob  = modelo_val.predict_proba(X_val)[0]
pred  = le_val.inverse_transform([np.argmax(prob)])[0]
print(f"Predição de teste: {pred} ({max(prob)*100:.1f}% de confiança)")
print("Modelo de Melhor Matéria validado!")


---
## 18. Modelo 2: Predição de Evasão (Churn)

### 🚪 O que é Churn no contexto da Passos Mágicos?

**Evasão** ocorre quando um aluno que estava ativo em 2022 não aparece nos dados de 2024.
Este modelo prevê, com base nos indicadores de 2022, quais alunos têm maior probabilidade de
abandonar o programa — permitindo **intervenção preventiva** pela equipe da ONG.

### 🔑 Estratégia anti-leakage

| Elemento | Decisão |
|---|---|
| **Base** | Alunos com dados em 2022 (último ano histórico) |
| **Target** | `CHURN = 1` se ausente em 2024, `0` se presente |
| **Features** | Indicadores de **2022** + proxies socioeconômicos do CSV |
| **Proibido** | Qualquer dado de 2024 como feature |

> ⚠️ **Nota:** Alunos que se formaram ou concluíram naturalmente o programa também aparecem
> como CHURN=1. Isso é uma limitação dos dados disponíveis, não do modelo.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 32 — Construção do Dataset de Churn
# ─────────────────────────────────────────────────────────────────────────

from train_model import criar_dataset_churn, FEAT_CHURN

df_churn, proxy_csv, cols_proxy = criar_dataset_churn(df_long, df_csv)

n0   = (df_churn["CHURN"] == 0).sum()
n1   = (df_churn["CHURN"] == 1).sum()
taxa = n1 / len(df_churn) * 100

print("=" * 55)
print("  Dataset de Churn/Evasão")
print("=" * 55)
print(f"  Total de alunos (base 2022): {len(df_churn)}")
print(f"  Retidos em 2024  : {n0} ({n0/len(df_churn)*100:.1f}%)")
print(f"  Evadiram        : {n1} ({taxa:.1f}%)")
print(f"  Features ({len(FEAT_CHURN)}): {FEAT_CHURN}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 33 — Visualização: Distribuição de Evasão e Perfil por Grupo
# ─────────────────────────────────────────────────────────────────────────

import plotly.express as px
import plotly.graph_objects as go

# ── Distribuição Churn ────────────────────────────────────────────────────
fig1 = px.pie(
    values=[n0, n1], names=['Retido (0)', 'Evadiu (1)'],
    color_discrete_map={'Retido (0)': '#4CAF50', 'Evadiu (1)': '#D84C51'},
    title='📊 Distribuição de Evasão (base 2022)',
    hole=0.4, template='plotly_white'
)
fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.show()

# ── Comparativo de indicadores: Retido vs Evadiu ─────────────────────────
inds = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPV', 'IAN']
medias = []
for ind in inds:
    medias.append({'Indicador': ind, 'Grupo': 'Retido',  'Média': df_churn[df_churn['CHURN']==0][ind].mean()})
    medias.append({'Indicador': ind, 'Grupo': 'Evadiu',  'Média': df_churn[df_churn['CHURN']==1][ind].mean()})

df_comp = pd.DataFrame(medias)
fig2 = px.bar(
    df_comp, x='Indicador', y='Média', color='Grupo', barmode='group',
    color_discrete_map={'Retido': '#4CAF50', 'Evadiu': '#D84C51'},
    title='📈 Média dos Indicadores: Retidos vs Evadiram',
    template='plotly_white', text_auto='.2f'
)
fig2.show()

print('💡 Insight: Alunos que evadiram apresentam, em média, menores índices de')
print('   engajamento (IEG) e psicossocial (IPS), o que orienta as features do modelo.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 34 — Treinamento do Modelo de Churn (XGBoost)
# Usa treinar_churn() de train_model.py (arquivo autônomo)
# ─────────────────────────────────────────────────────────────────────────

from train_model import treinar_churn

print("Treinando Modelo de Churn (XGBoost)...")
art_churn = treinar_churn()
res = art_churn["resultados"]
print("=" * 55)
print("  Resultados (5-Fold CV)")
print("=" * 55)
print(f"  CV Acurácia : {res["cv_acc_mean"]*100:.1f}% ± {res["cv_acc_std"]*100:.1f}%")
print(f"  CV F1-Score : {res["cv_f1_mean"]*100:.1f}% ± {res["cv_f1_std"]*100:.1f}%")
print(f"  CV AUC-ROC  : {res["cv_auc_mean"]:.3f} ± {res["cv_auc_std"]:.3f}")
print(f"  Train Acc   : {res["train_acc"]*100:.1f}%")
print(f"  Gap Overfit : {res["overfitting_gap"]:.3f}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 35 — Avaliação: Feature Importance e CV por Fold
# ─────────────────────────────────────────────────────────────────────────

# ── Feature Importance ───────────────────────────────────────────────────
fig_imp = px.bar(
    modelo_churn.feature_importance,
    x='Importância', y='Feature', orientation='h',
    color='Importância', color_continuous_scale='Reds',
    title='🎯 Importância das Features — Modelo de Churn',
    template='plotly_white'
)
fig_imp.update_layout(coloraxis_showscale=False)
fig_imp.show()

# ── CV por Fold ──────────────────────────────────────────────────────────
cv_df = pd.DataFrame({
    'Fold':     [f'Fold {i+1}' for i in range(5)],
    'Acurácia': res['cv_acc_scores'],
    'F1-Score': res['cv_f1_scores'],
})
fig_cv = px.bar(
    cv_df.melt(id_vars='Fold', var_name='Métrica', value_name='Score'),
    x='Fold', y='Score', color='Métrica', barmode='group',
    color_discrete_map={'Acurácia': '#2D325E', 'F1-Score': '#EE8133'},
    title='📈 Performance por Fold — 5-Fold CV (Churn)',
    template='plotly_white', text_auto='.1%'
)
fig_cv.show()

top = modelo_churn.feature_importance.iloc[-1]['Feature']
print(f'💡 O fator mais determinante para a evasão é: {top}')
print(f'   Threshold calibrado via CV: {modelo_churn.threshold:.2f}')
print(f'   (Valores de probabilidade acima deste threshold → aluno classificado como EVADIU)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 36 — Predição Individual de Evasão
# ─────────────────────────────────────────────────────────────────────────

# Exemplo de predição para um aluno hipotético com perfil de risco
aluno_risco = {
    'INDE': 5.2, 'IAA': 4.8, 'IEG': 4.0, 'IPS': 4.5,
    'IDA': 5.0, 'IPV': 4.2, 'IAN': 5.0,
    'ESCOLA_PUBLICA': 1, 'BOLSISTA': 0, 'IDADE': 16,
    'ANOS_PM': 2, 'DEFASAGEM_ABS': 2, 'HISTORICO_ANOS': 1, 'NUM_AVALIADORES': 2,
}

# Aluno com perfil de retenção
aluno_retido = {
    'INDE': 8.1, 'IAA': 8.5, 'IEG': 8.0, 'IPS': 8.3,
    'IDA': 8.2, 'IPV': 7.8, 'IAN': 8.5,
    'ESCOLA_PUBLICA': 0, 'BOLSISTA': 1, 'IDADE': 14,
    'ANOS_PM': 4, 'DEFASAGEM_ABS': 0, 'HISTORICO_ANOS': 2, 'NUM_AVALIADORES': 3,
}

pred_risco  = modelo_churn.predizer_risco_evasao(aluno_risco)
pred_retido = modelo_churn.predizer_risco_evasao(aluno_retido)

print('─' * 50)
print('  Aluno com PERFIL DE RISCO:')
print(f"  Resultado     : {pred_risco['classificacao']}")
print(f"  Prob. Evasão  : {pred_risco['probabilidade_evasao']*100:.1f}%")
print(f"  Threshold     : {pred_risco['threshold_usado']:.2f}")
print()
print('  Aluno com PERFIL DE RETENÇÃO:')
print(f"  Resultado     : {pred_retido['classificacao']}")
print(f"  Prob. Evasão  : {pred_retido['probabilidade_evasao']*100:.1f}%")
print('─' * 50)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CÉLULA 37 — Verificação do Modelo de Churn Salvo
# O modelo já foi salvo por treinar_churn() na célula anterior.
# ─────────────────────────────────────────────────────────────────────────

import os, joblib

# Validação
art_val = joblib.load("models/modelo_churn.joblib")
modelo_val = art_val["modelo"]
feats_val  = art_val["features"]
thr_val    = art_val["threshold"]

aluno_risco = {
    "INDE": 5.2, "IAA": 4.8, "IEG": 4.0, "IPS": 4.5,
    "IDA": 5.0, "IPV": 4.2, "IAN": 5.0,
    "ESCOLA_PUBLICA": 1, "BOLSISTA": 0, "IDADE": 16,
    "ANOS_PM": 2, "DEFASAGEM_ABS": 2, "HISTORICO_ANOS": 1, "NUM_AVALIADORES": 2,
}
import numpy as np
X_val = np.array([[aluno_risco.get(f, 0) for f in feats_val]])
prob  = modelo_val.predict_proba(X_val)[0][1]
pred  = "EM RISCO DE EVASÃO" if prob >= thr_val else "PERMANÊNCIA PROVÁVEL"
print(f"Predição de teste: {pred} (prob={prob:.3f}, thr={thr_val:.2f})")
print("Modelo de Churn validado!")


---
## 17. Conclusões e Próximos Passos

### 🏆 Sumário Executivo

Este projeto entregou **um sistema end-to-end de predição de risco de defasagem** para a ONG Passos Mágicos, cobrindo desde a ingestão dos dados até uma aplicação interativa em produção.

### 📊 Principais Achados Analíticos

| Encontrado | Implicação |
|------------|------------|
| **IEG** é forte preditor do IPV | Estimular engajamento é a alavanca mais efetiva para o ponto de virada |
| Alunos com **IPS baixo** têm IDA menor | Suporte psicossocial tem impacto direto no aprendizado |
| **IAA** tende a superestimar o IDA | Alunos podem não perceber dificuldades reais → sinaliza necessidade de feedback |
| **IPP** e **IAN** fortemente correlacionados | Avaliação profissional está bem alinhada com métricas objetivas |
| Maior **tempo na PM** → melhores indicadores | Permanência no programa tem efeito cumulativo positivo |

### 🤖 Resultado do Modelo

- **Melhor algoritmo:** identificado automaticamente por F1-Score no test set
- **Validação cruzada:** 5-fold estratificado garante robustez
- **Anti-leakage:** apenas features que um educador observa *antes* de conhecer a defasagem

### 🚀 Próximos Passos Recomendados

1. **Dados Socioeconômicos:** Integrar CEP + bases públicas (ex: IBGE, IPEA) para criar proxies socioeconômicos (renda do bairro, IDH) como features adicionais  
2. **Análise de Trajetória Individual:** Usar variação temporal dos indicadores (delta ano-a-ano) como features  
3. **SHAP Values:** Implementar explicabilidade por aluno para orientar intervenções específicas  
4. **Threshold Otimizado:** Ajustar o limiar de decisão para maximizar recall (evitar falsos negativos)  
5. **Deploy em Produção:** A aplicação Streamlit (app.py) já está pronta — integrar com o banco de dados oficial da PM  
6. **Monitoramento:** Implementar drift detection para detectar quando o modelo precisa ser re-treinado

---

### 📚 Arquitetura do Código

```
src/data_loader.py
├── carregar_dataset_csv()        — Ingestão e limpeza do CSV 2020-2022
├── carregar_dataset_xlsx()       — Ingestão e padronização do XLSX 2024  
├── criar_dataset_longitudinal()  — Transforma wide → long (formato de séries temporais)
├── criar_dataset_modelo()        — Engenharia de features + anti-leakage + target
├── classificar_risco_ian()       — Classifica risco por IAN (4 categorias)
└── classificar_pedra()           — Classifica por pedra via INDE (4 categorias)

src/analysis.py
├── analise_ian()                 — Pergunta 1: Defasagem geral (IAN)
├── analise_ida()                 — Pergunta 2: Desempenho acadêmico
├── analise_engajamento()         — Pergunta 3: IEG × IDA × IPV
├── analise_autoavaliacao()       — Pergunta 4: IAA vs realidade
├── analise_psicossocial()        — Pergunta 5: IPS e seus impactos
├── analise_psicopedagogico()     — Pergunta 6: IPP vs IAN
├── analise_ponto_virada()        — Pergunta 7: Drivers do IPV
├── analise_multidimensional()    — Pergunta 8: Correlações + Radar
├── analise_efetividade()         — Pergunta 9/10: Efetividade do programa
└── analise_insights()            — Pergunta 11: Gênero, Fase, Bolsa

src/model.py → class ModeloRiscoDefasagem
├── preparar_dados()              — Split estratificado + normalização
├── treinar_modelos()             — RF, XGBoost, GBM, LR com CV
├── obter_comparacao_modelos()    — DataFrame comparativo de métricas
├── predizer_risco()              — Predição individual em tempo real
├── salvar_modelo()               — Serialização via joblib
└── carregar_modelo()             — Deserialização do modelo salvo

train_model.py                    — Script CLI de treinamento
app.py                            — Aplicação Streamlit (dashboard interativo)
```

---
*Notebook criado pelo Analista de Dados — Projeto Datathon FIAP × Passos Mágicos — Abril 2026*